# Introduction to Scikit-Learn (sklearn)

This notebook demonstrates some of the most useful functions of the
beautiful Scikit-Learn library.

What we're going to cover:

0. An end-to-end Scikit-Learn workflow
1. Getting the data ready
2. Choose the right estimator (model) / algorithm for our problems
3. Fit the model/algorithm and use it to make predictions on our data
4. Evaluating a model
5. Improve a model
6. Save and load a trained model
7. Putting it all together

## 0. An end-to-end Scikit-Learn workflow

In [ ]:
# 1. Get the data ready
import pandas as pd
import numpy as np
import sklearn

heart_disease = pd.read_csv('./data/heart-disease.csv')
heart_disease

In [ ]:
# Create `X` (the "feature matrix")
# AKA data or feature variables
X = heart_disease.drop('target', axis=1)
X

In [ ]:
# Create the y (AKA labels or label matrix)
y = heart_disease['target']
y

## 2. Choose the right model and hyperparameters

In [ ]:
# Remember, "hyperparameters" are like "dials" we can use to (fine) tune our model
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier()

# We'll keep the default hyperparameters
clf.get_params()

## 3. Fit the model to the training data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
clf.fit(X_train, y_train);

In [ ]:
# Make a (faulty) prediction (using **incorrectly shaped** data)
try:
    y_broken_label = clf.predict(np.array([0, 2, 3, 4]))
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
# Make a (correct) prediction (using **correctly shaped** data)
# Note: we are still on step 3 or our workflow
y_label = clf.predict(X_test)

In [ ]:
y_preds = clf.predict(X_test)
y_preds

In [ ]:
y_test

## 4. Evaluate the model on the training data ...

In [ ]:
clf.score(X_train, y_train)

In [ ]:
# ... and on the test data
clf.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(classification_report(y_test, y_preds))  # clf.predict(X_test)

See the article, [Understanding a Classification Report](https://medium.com/@kohlishivam5522/understanding-a-classification-report-for-your-machine-learning-model-88815e2ce397),
for an explanation of this report.

In [ ]:
# Calculate the "confusion matrix" for test and predicted values
confusion_matrix(y_test, y_preds)

In [ ]:
accuracy_score(y_test, y_preds)

## 5. Improve a model

In [ ]:
# - Generate a one-time 128-bit secret for the seed.
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **Copy and paste** this value as the **hard-coded** random number
# generator seed
rng = np.random.default_rng(seed=77708057215789171477656921825058000355)

In [ ]:
# Try different amount of `n_estimators`
for i in range(10, 100, 10):
    print(f'Trying model with {i} estimators...')
    # I use the `default_rng` with a specified seed to generate reproducible results.
    # The maximum integer I want is the largest unsigned 32-bit integer. (This value
    # is the largest value compatible with the argument to `rng.integers`.)
    clf = RandomForestClassifier(
        n_estimators=i,
        random_state=rng.integers(np.iinfo(np.uint32).max)).fit(X_train, y_train)
    print(f'Model accuracy on test set: {clf.score(X_test, y_test) * 100:.2f}%')
    print('')  # simply to separate runs

The maximum accuracy of the test set, 91.80%, occurs with 50 estimators.

Consequently, we can **improve** our model by using 50 estimators.

## 6. Save and load a trained model

In [ ]:
# We can save a model using `pickle`.
import pickle

In [ ]:
# The video calls `pickle.dump(clf, open('random_forest_model.pkl', 'wb'))`.
# This call results in a type warning similar to
# "Expected SupportsWrite but got BinaryIO".
# This situation is resolved by using `with` below.

with open('./random_forest_model_1.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [ ]:
# What happens if we try to import the saved model.
with open('./random_forest_model_1.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [ ]:
loaded_model.score(X_test, y_test)

This result is the **same** as the last model we tried (90 estimators). **Hooray!**

In [ ]:
# Let's "listify" the contents
what_were_covering = [
    '0. An end-to-end Scikit-Learn workflow',
    '1. Getting the data ready',
    '2. Choose the right estimator/algorithm/model for your problem',
    '3. Fitting your chosen machine learning model to data and using it to make a prediction',
    '4. Evaluating a machine learning model',
    '5. Improving predictions through experimentation (hyperparameter tuning)',
    '6. Saving and loading a pre-trained model',
    '7. Putting it all together in a pipeline',
]

In [ ]:
what_were_covering

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# %matplotlib inline

## 1. Getting our data ready to be used with machine learning

Three main tasks to complete:

1. Split the data into features and labels (usually `X` and `y`)
2. Filling (AKA imputing) or disregarding missing values
3. Converting non-numerical values to numerical values (also called "feature encoding")

In [ ]:
heart_disease.head()

In [ ]:
# The last column is our data to be predicted so we drop it
# Remember that `axis=1` is the **column** axis
# (`axis=0` is the **row** axis)
X = heart_disease.drop('target', axis=1)
X.head()

In [ ]:
y = heart_disease['target']
y.head()

In [ ]:
# Split the features and labels into training and test splits
# We reserve 20% of our data for testing. This amount is "negotiable" for
# different problems.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X.shape

In [ ]:
len(heart_disease)

In [ ]:
len(heart_disease) * 0.8

In [ ]:
242 + 61

In [ ]:
len(heart_disease)

### 1.1 Make sure all data is **numerical**

In [ ]:
car_sales = pd.read_csv('./data/car-sales-extended.csv')
car_sales.head()

In [ ]:
car_sales['Doors'].value_counts()

Notice that `car_sales['Doors']` is **both** numeric **and** categorical.

It is numeric because its values are integers. But it is categorical
because it's (mathematical) range is only a small subset of integers.

As a consequence of the small subset of values, we will treat this column
as a **categorical** column (see our encoding code later).

In [ ]:
len(car_sales)

In [ ]:
car_sales.dtypes

In [ ]:
# Split into `X` and `y`
X = car_sales.drop('Price', axis=1)
y = car_sales['Price']

# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Build machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor() ## Create our model
try:
    model.fit(X_train, y_train) ## Fit our model
    model.score(X_test, y_test) ## Score our model
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')
# Score

Remember, we **must** convert strings (objects) to **numbers**

In [ ]:
# Turn the (object) categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify features by **column names**
categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot',  ## Name our transform
                                  one_hot,  ## The specific transformer
                                  categorical_features)],  ## Applies **only** to `categorical_features`
                                remainder='passthrough')  ## Pass all other columns **unchanged**
transformed_X = transformer.fit_transform(X)
transformed_X

In [ ]:
pd.DataFrame(transformed_X)

In [ ]:
# An alternative to one-hot encoding
dummies = pd.get_dummies(car_sales[['Make', 'Colour', 'Doors']])
dummies

Now that our data is all numeric (zeros and ones), let's refit the model

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=22232115356560702892793270496072236204)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
print(sklearn.__version__)

### 1.2 What if I find **missing** data?

1. Fill them with some value (AKA imputation)
2. Remove the samples with missing data altogether

Neither of these techniques is "perfect" or "recommended"
- Replacing "missing" data might introduce bias or "throw away" "significant" information
- Removing samples completely results in less total data to use

In [ ]:
# Import car sales with missing data
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Extract `X` and `y` (features and values)
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Calling `.sum()` takes advantage of Python "coercion" of Boolean types
# That is, True is converted to 1 when summing and False is converted to 0.
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

At this point in the video, Python reports an exception:
"ValueError: Input contains NaN"

Because I'm using `sklearn` version 1.5.x, I **do not** see this error.

But I'll pretend like I do.

In [ ]:
car_sales_missing

In [ ]:
car_sales_missing['Doors'].value_counts()

In [ ]:
car_sales_missing['Doors'].mode()

#### Option 1: Fill missing data with `pandas`

In [ ]:
# Fill the "Make" column
car_sales_missing['Make'] = car_sales_missing['Make'].fillna('missing')

# Fill the "Colour" column
car_sales_missing['Colour'] = car_sales_missing['Colour'].fillna('missing')

# Fill the "Odometer (KM)" column
car_sales_missing['Odometer (KM)'] = car_sales_missing['Odometer (KM)'].fillna(car_sales_missing['Odometer (KM)'].mean())

# Fill the "Doors" column
# A little tricky because this column is actually a **categorical** column.
# Because the (overwhelming) majority of cars have 4 doors, we will
# replace all missing values in the 'Doors' column with the value 4.
# (Because 4 is the most common value, we could replace the hard-coded
# value of 4 with `car_sales_missing['Doors'].mode()`
car_sales_missing['Doors'] = car_sales_missing['Doors'].fillna(4)

In [ ]:
# Check our `DataFrame` again
car_sales_missing.isna().sum()

In [ ]:
# Because 'Price' is our value column, we **do not** want to replace
# missing values with another value. Instead, we will **remove** all
# rows that are missing a value in the 'Price' column.
car_sales_missing = car_sales_missing.dropna(subset=['Price'])

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
len(car_sales_missing)

In [ ]:
# Remember, one must **always** split data into `X` and `y`
# (features and labels) after **changing** the data.
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

#### Option 2. Fill missing values with Scikit-Learn

In [ ]:
# Read data (as usual)
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Check for missing data
car_sales_missing.isna().sum()

In [ ]:
# Remove rows **without** labels
car_sales_missing = car_sales_missing.dropna(subset=['Price'])
car_sales_missing.isna().sum()

In [ ]:
# Split into features and labels
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
X.isna().sum()

In [ ]:
# Fill missing values from Scikit-LearnA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Fill categorical values with 'missing' and numerical values with `mean()`
categorical_imputer = SimpleImputer(strategy='constant', fill_value='missing')
door_imputer = SimpleImputer(strategy='constant', fill_value=4)
numeric_imputer = SimpleImputer(strategy='mean')

# Define columns
categorical_features = ['Make', 'Colour']
door_feature = ['Doors']  ## Because 'Doors' column is a special case
numeric_features = ['Odometer (KM)']

# Create an imputer (that is, something the fills missing data)
imputer = ColumnTransformer([
    ('categorical_features', categorical_imputer, categorical_features),
    ('door_feature', door_imputer, door_feature),
    ('numeric_features', numeric_imputer, numeric_features),
])

# (Finally) Transform the features
filled_X = imputer.fit_transform(X)
filled_X

In [ ]:
# Check our code as we have done previously (using `isna().sum()`
car_sales_missing = pd.DataFrame(filled_X,
                                 columns=['Make', 'Colour', 'Doors', 'Odometer (KM)'])
car_sales_missing

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

Now our data is

- All numeric
- Filled (no missing values)

Let's fit a model!

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=124608693003265431754472593407374562997)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
len(car_sales_missing), len(car_sales)

**Note**: The 50 less values in the transformed data is because we
dropped the rows (50 total) with missing values in the
transformed data.

In [ ]:
what_were_covering

## 2. Choosing the right estimator/algorithm/model for your problem

Some things to note:

- The package, `sklearn`, refers to machine learning models and
  algorithms as _estimators_
  - A _classifier_ is one type of estimator
  - A _regressor_ is another type of estimator
- Classification problem - predicting a **category**
  - For example, heart disease or not heart disease
  - Sometimes you see the term, `clf`
    - A TLA for **classifier**
    - Used as a classification estimator
- Regression problem - predicting a **number**
  - For example, the selling price of a car

If you're

- Working on a machine learning problem
- Looking to use `sklearn`
- Unsure what model you should use

Use the `sklearn` [machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html)


### 2.1 Picking a machine learning model for a regression problem

Let's use the California Housing Data Set

Daniel, where are you getting these ideas from?

The `sklearn` documentation on [datasets](https://scikit-learn.org/stable/datasets.html)

- [Toy datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [Real World datasets](https://scikit-learn.org/stable/datasets/real_world.html)
- [Generated datasets](https://scikit-learn.org/stable/datasets/sample_generators.html)
- [Loading other datasets](https://scikit-learn.org/stable/datasets/loading_other_datasets.html)


In [ ]:
# Get the California Housing dataset
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
housing

In [ ]:
# Let's turn our data into a `pandas` `DataFrame`
# Remember that the **datat** is in `housing.data`
housing_df = pd.DataFrame(housing['data'],
                          columns=housing['feature_names'])
housing_df

In [ ]:
# We must add our target data to the data frame
housing_df[housing.target_names[0]] = housing.target
housing_df

In [ ]:
# Lets change the target column name to "target"
housing_df = housing_df.rename({'MedHouseVal': 'target'}, axis=1)
housing_df

In [ ]:
# Split data into features and targets (`X` and `y`)
X = housing_df.drop('target', axis=1)
y = housing_df['target']

In [ ]:
X

In [ ]:
y

Let's use our "full" set of steps

- Import algorithm
- Setup random seed
- Create the data

In [ ]:
# Import algorithm
from sklearn import linear_model

In [ ]:
# Set up repeatable random number generator part 1

# Calculate our new seed
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# Set up repeatable random number generator part 2

# Create a random number generator with the specified seed
rng = np.random.default_rng(seed=187614292792846192168046308353096246446)

In [ ]:
# Create the data
X = housing_df.drop('target', axis=1)
y = housing_df['target'] # median house price in $100k

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate and fit the model on the **training** set
model = linear_model.Ridge(alpha=0.5)
model.fit(X_train, y_train)

# Check the score of the model on the test set
model.score(X_test, y_test)

What if `Ridge` did not work or the score did not fit our needs?

Well, we could always try a different model....

Let's try an "ensemble model"!

An "ensemble" is a combination of smaller models to try to make
better predictions than just a single model.

The ensemble models for `sklearn` can be found [here](https://scikit-learn.org/stable/modules/ensemble.html)

In [ ]:
# Import the `RandomForestRegressor` model class from the ensemble module
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Calculate our new random seed
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# Use this seed to seed a "new" random number generator
rng = np.random.default_rng(seed=20717103177106046274233922339781200557)

In [ ]:
# Create the data
X = housing_df.drop('target', axis=1)
y = housing_df['target']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create, fit, and score our random forest model
model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)


### 2.2 Choosing an estimator for a classification problem

Let's go to the map... [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html)

In [ ]:
heart_disease = pd.read_csv('./data/heart-disease.csv')
heart_disease

Consulted the [Online machine learning model map]if (https://scikit-learn.org/stable/machine_learning_map.html):
It's recommendation: try a "LinearSVC" model.

In [ ]:
# Import the classifier (and other required libraries)
from sklearn.svm import LinearSVC

In [ ]:
# Generate a problem-specific seed
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=4577550920016562965326478789717866486)

In [ ]:
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = LinearSVC() ## `clf` is a TLA for "classifier"
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
heart_disease['target'].value_counts()

In [ ]:
X

K-neighbors classifier

The package, `sklearn.neighbors`, "...provides functionality for
unsupervised and supervised neighbors-based learning methods.

Unfortunately, after reading the documentation of

- `NearestNeighbors`
- `KDTree`
- `BallTree`

I was unclear how to understand its application to our problem.
So... back to the video

Remember, when we previously moved to `RandomForestRegressor`, we
saw an **improvement** in our model score. Hopefully, we will see
an improvement using a `RandomForestClassifier`.

An interesting note from the [documentation](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)

> The purpose of these two sources of randomness is to decrease the variance
> of the forest estimator. Indeed, individual decision trees typically
> exhibit high variance and tend to overfit. The injected randomness in
> forests yield decision trees with somewhat decoupled prediction errors.
> By taking an average of those predictions, some errors can cancel out.
> Random forests achieve a reduced variance by combining diverse trees,
> sometimes at the cost of a slight increase in bias. In practice the
> variance reduction is often significant hence yielding an overall
> better model.



In [ ]:
# Import the classifier (and other required libraries)
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Generate a problem-specific seed
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=6318142287573278828684754060508691815)
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = RandomForestClassifier()  ## `clf` is a TLA for "classifier"
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

A machine learning tidbit from Daniel

> If you have structured data - AKA tables of data frames - use
> ensemble methods.
>
> Why? Because it will perform pretty well.
>
> If there are patterns

Tidbit (or shortcut):

| If you have... | Then use |
| -------------- | -------- |
| Structured data | Ensemble methods |
| Unstructured data | Deep learning methods |

Data Examples

| Structured Data | Unstructured Data |
| --------------- | ----------------- |
| Data in a table like `heart_disease` | - Images |
|                                      | - Audio |
|                                      | - Text |


In [ ]:
heart_disease

Remember, a critical success factor in data science is

- **Reducing** your time between experiments

In [ ]:
what_were_covering

## 3. Fit the model/algorithm to our data and use it to make predictions

### 3.1. Fitting the model to the data

Common "aliases"

- `X` - features, feature variables, data
- `y` - labels, targets, target values

In [ ]:
# Import the classifier (and other required libraries)
from sklearn.ensemble import RandomForestClassifier
# Generate a problem-specific seed

In [ ]:
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=2075748097716160523040509094081475903)

In [ ]:
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = RandomForestClassifier()  ## `clf` is a TLA for "classifier"

# Fit the data
clf.fit(X_train, y_train)

# Evaluate the `RandomForestClassifier`
clf.score(X_test, y_test)

In [ ]:
X.head()

In [ ]:
y.head(), y.tail()

### Random Forest model deep dive

See [Random Forest Resources](notes/complete-ai-ml-ds/Random%20Forest%20Resources.md)
for more information.

### 3.2 Make predictions using a machine learning model

Two main ways to make predictions

- `predict()`
- `predict_proba()`

In [ ]:
# Use a trained model to make predictions

# We'll start with a mistake

try:
    clf.predict(np.array([2, 7, 1, 7, 2, 8])) # This choice **does not** work
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
clf.predict(X_test)

In [ ]:
np.array([y_test]).shape, clf.predict(X_test).shape

In [ ]:
# To evaluate our model, compare **predictions** to the
# "true" labels.
y_predictions = clf.predict(X_test)
np.mean(y_predictions == y_test)

In [ ]:
# This value is the same as...
clf.score(X_test, y_test)

In [ ]:
# We're getting ahead of ourselves, but...
# (Another way to calculate our score)
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_predictions)

In [ ]:
what_were_covering

Make predictions using `predict_proba()`

In [ ]:
# Note that `predict_proba()` returns the probabilities of a
# classification label.
clf.predict_proba(X_test)

In [ ]:
clf.predict_proba(X_test[:5])

In [ ]:
# Lets use `predict()` on the same data...
clf.predict(X_test[:5])

In [ ]:
X_test[:5]

In [ ]:
# Notice
0.89 + 0.11